# Stochastic Calculus from Zero · Episode 24: The reflection principle and barrier options

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> **The share** (Series 3's): priced at 100, 20% volatility, a 3.60% risk-free rate (continuous), a call with strike 105 expiring in a year, and an illustrative real-world drift of 8%. Made up for teaching. The barrier, 130, is illustrative too.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. Reflect after the first hit · 4. The formula · 5. Simulated: grids miss crossings · 6. A barrier option · 7. Daily checks versus the Brownian bridge · 8. This episode's question · 9. Exercises · 10. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 24,
    "quantlib_version": ql.__version__,
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, math, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)


# ---- Part 3 additions -------------------------------------------------------------------------------------------
#   Short rate (illustrative): r0 3.85%, long-run level 3.50%, mean reversion 0.30, volatility 1% (Vasicek).
def norm_cdf(x):
    """Standard normal distribution function, for numbers or numpy arrays."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x) / math.sqrt(2)))


def bs_call(s, k, t, r, vol):
    """Black-Scholes call price and delta (Series 3, Episode 13)."""
    d1 = (np.log(s / k) + (r + 0.5 * vol ** 2) * t) / (vol * np.sqrt(t))
    return s * norm_cdf(d1) - k * np.exp(-r * t) * norm_cdf(d1 - vol * np.sqrt(t)), norm_cdf(d1)


def ou_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of an Ornstein-Uhlenbeck (Vasicek) rate t years ahead."""
    return theta + (r0 - theta) * math.exp(-a * t), sigma * math.sqrt((1 - math.exp(-2 * a * t)) / (2 * a))


def ou_step(r, dt, a, theta, sigma, z):
    """One exact OU step of length dt from r, driven by standard normals z: no discretisation error."""
    return theta + (r - theta) * math.exp(-a * dt) + sigma * math.sqrt((1 - math.exp(-2 * a * dt)) / (2 * a)) * z


def vasicek_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price P = A exp(-B r) in the Vasicek model, tau years to maturity."""
    B = (1 - math.exp(-a * tau)) / a
    lnA = (theta - sigma ** 2 / (2 * a * a)) * (B - tau) - sigma ** 2 * B * B / (4 * a)
    return np.exp(lnA - B * np.asarray(r))


def cir_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of a CIR rate t years ahead."""
    e = math.exp(-a * t)
    var = r0 * sigma ** 2 / a * (e - e * e) + theta * sigma ** 2 / (2 * a) * (1 - e) ** 2
    return theta + (r0 - theta) * e, math.sqrt(var)


def cir_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price in the CIR model (Cox, Ingersoll and Ross, 1985)."""
    g = math.sqrt(a * a + 2 * sigma * sigma)
    den = (g + a) * (math.exp(g * tau) - 1) + 2 * g
    B = 2 * (math.exp(g * tau) - 1) / den
    A_ = (2 * g * math.exp((a + g) * tau / 2) / den) ** (2 * a * theta / sigma ** 2)
    return A_ * np.exp(-B * np.asarray(r))

## 2. Last episode's answer

With $\sigma = 0$: $P = \exp\big(-\theta T - (r_0 - \theta)(1 - e^{-aT})/a\big) = 0.8319$, about 6.5 basis points of price below the bond with volatility.

In [ ]:
out["prev"] = {"answer": math.exp(-0.035 * 5 - (0.0385 - 0.035) * (1 - math.exp(-1.5)) / 0.3)}

## 3. Reflect after the first hit

Let $M_T = \max_{t \le T} W_t$, the **running maximum**, and $m > 0$ a level. Take a path that reaches $m$, at the first time $\tau$. From $\tau$ on, the path's future increments are a fresh Brownian motion (the strong Markov property), symmetric about 0: mirror them in the line $y = m$ and you get an equally likely path. So among paths that hit $m$, those ending below $m$ pair up one-to-one with those ending above it.

In [ ]:
rng = np.random.default_rng(seed)
m_level, steps = 1.0, 1_000
t = np.linspace(0, 1, steps + 1)
while True:                                       # first simulated path that hits 1 before mid-year and ends below it
    w = np.concatenate([[0.0], np.cumsum(rng.standard_normal(steps) * math.sqrt(1 / steps))])
    hit = np.flatnonzero(w >= m_level)
    if hit.size and hit[0] < steps // 2 and -0.2 < w[-1] < m_level - 0.3:
        break
tau = hit[0]
mirror = w.copy(); mirror[tau:] = 2 * m_level - w[tau:]
out["mirror"] = {"t": t[::4].round(4).tolist(), "path": [w[::4].round(4).tolist()], "mirror": mirror[::4].round(4).tolist(),
                 "tau": float(t[tau]), "end": float(w[-1]), "mirror_end": float(mirror[-1]), "m": m_level,
                 "ticks": [{"x": x, "label": lab} for x, lab in ((0, "0"), (0.25, "3m"), (0.5, "6m"), (0.75, "9m"), (1, "1y"))]}
{k: v for k, v in out["mirror"].items() if not isinstance(v, list)}

## 4. The formula

Paths with $M_T \ge m$ split into those with $W_T \ge m$ and those with $W_T < m$; by reflection the two groups are equally likely. So

$$P(M_T \ge m) = 2\,P(W_T \ge m) = 2\big(1 - N(m/\sqrt T)\big).$$

The maximum of a Brownian motion over a year is distributed like $|W_1|$.

In [ ]:
theory = 2 * (1 - float(norm_cdf(m_level)))
out["reflect"] = {"theory": theory, "theory_pct": theory * 100, "end_above_pct": (1 - float(norm_cdf(m_level))) * 100}
out["reflect"]

## 5. Simulated: grids miss crossings

20,000 paths on a daily grid (252 steps) and on a fine grid (4,096). A path can cross 1 and come back between two grid points, so a grid always undercounts.

In [ ]:
def frac_max_above(level, n_paths, steps, chunk=2_000):
    hits = 0
    for _ in range(n_paths // chunk):
        W = np.cumsum(rng.standard_normal((chunk, steps)) * math.sqrt(1 / steps), axis=1)
        hits += int((W.max(axis=1) >= level).sum())
    return hits / n_paths

sim = {"paths": 20_000, "daily": frac_max_above(1.0, 20_000, 252), "fine": frac_max_above(1.0, 20_000, 4096)}
sim["daily_pct"], sim["fine_pct"] = sim["daily"] * 100, sim["fine"] * 100
assert sim["daily"] < sim["fine"] < theory + 0.01 and abs(sim["fine"] - theory) < 0.02
out["sim"] = sim
sim

## 6. A barrier option

An **up-and-out call**: the strike-105 call, but it dies (pays nothing) if the share ever trades at or above 130 during the year, checked continuously. Applying the reflection principle to the drifted log price gives a closed form (Merton 1973; Reiner and Rubinstein 1991), which QuantLib's `AnalyticBarrierEngine` implements.

In [ ]:
S0, VOL, R, K, T, BARRIER = 100.0, 0.20, 0.036, 105.0, 1.0, 130.0
today = ql.Date(29, 9, 2026); ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
flat = lambda x: ql.YieldTermStructureHandle(ql.FlatForward(today, x, dc, ql.Continuous))
proc = ql.BlackScholesMertonProcess(ql.QuoteHandle(ql.SimpleQuote(S0)), flat(0.0), flat(R),
                                    ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), VOL, dc)))
def ql_barrier(kind, level):
    o = ql.BarrierOption(kind, level, 0.0, ql.PlainVanillaPayoff(ql.Option.Call, K), ql.EuropeanExercise(today + 365))
    o.setPricingEngine(ql.AnalyticBarrierEngine(proc))
    return o.NPV()
barrier = {"level": BARRIER, "K": K, "ql_up_out": ql_barrier(ql.Barrier.UpOut, BARRIER), "vanilla": float(bs_call(S0, K, T, R, VOL)[0])}
barrier

## 7. Daily checks versus the Brownian bridge

Monte Carlo, 100,000 risk-neutral paths with daily steps. **Daily monitoring**: kill a path if it is at or above 130 at a daily close. That misses crossings between closes, so it overprices a continuously monitored knock-out. **Brownian-bridge correction**: given the log price at two closes $x$ and $y$, both below the log barrier $b$, the chance the path crossed in between is

$$p = \exp\!\Big(-\frac{2(b - x)(b - y)}{\sigma^2\Delta t}\Big),$$

so weight each path by the product of $(1 - p)$ over its steps.

In [ ]:
n_paths, n = 100_000, 252
dt = T / n
x = np.full(n_paths, math.log(S0)); alive = np.ones(n_paths, bool); survive = np.ones(n_paths)
lb = math.log(BARRIER)
for _ in range(n):
    xn = x + (R - 0.5 * VOL ** 2) * dt + VOL * math.sqrt(dt) * rng.standard_normal(n_paths)
    alive &= xn < lb
    p_cross = np.where((x < lb) & (xn < lb), np.exp(-2 * (lb - x) * (lb - xn) / (VOL ** 2 * dt)), 1.0)
    survive *= 1 - p_cross
    x = xn
pay = math.exp(-R * T) * np.maximum(np.exp(x) - K, 0)
mc = {"paths": n_paths, "daily": float((pay * alive).mean()), "bridge": float((pay * survive).mean()),
      "bridge_se": float((pay * survive).std() / math.sqrt(n_paths)), "daily_se": float((pay * alive).std() / math.sqrt(n_paths)),
      "share_knocked_daily": float(1 - alive.mean())}
mc["overprice_pct"] = (mc["daily"] / barrier["ql_up_out"] - 1) * 100
assert abs(mc["bridge"] - barrier["ql_up_out"]) < 3 * mc["bridge_se"]
assert mc["daily"] > mc["bridge"]
out["barrier"] = barrier
out["mc"] = mc
pd.Series({**barrier, **mc})

## 8. This episode's question

$P(\max_{t \le 1} W_t \ge 2) = 2(1 - N(2)) = 4.55\%$.

In [ ]:
out["question"] = {"m": 2, "p_pct": 2 * (1 - float(norm_cdf(2.0))) * 100}
out["question"]

## 9. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** What is the chance that a Brownian motion reaches 0.5 within a year?

<details><summary>Answer</summary>

$2(1 - N(0.5)) = 2 \times 0.3085 = 0.617$.

</details>

**Exercise 2 (By hand).** What is the chance that $W$ stays below 1 for four years?

<details><summary>Answer</summary>

$1 - 2(1 - N(1/\sqrt4)) = 1 - 2(1 - N(0.5)) = 0.383$.

</details>

**Exercise 3 (By hand).** Why is an up-and-in call plus an up-and-out call (same strike, same barrier) worth exactly the vanilla call?

<details><summary>Answer</summary>

On every path exactly one of them is alive at expiry and it pays $\max(S_T - K, 0)$; the other pays 0. Same payoff on every path, same price.

</details>

**Exercise 4 (In code).** Broadie, Glasserman and Kou (1997): a barrier checked every $\Delta t$ is priced well by the continuous formula with the barrier moved away by $e^{0.5826\,\sigma\sqrt{\Delta t}}$. Compare with the daily Monte Carlo.

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
shifted = BARRIER * math.exp(0.5826 * VOL * math.sqrt(dt))
bgk = ql_barrier(ql.Barrier.UpOut, shifted)
assert abs(bgk - mc["daily"]) < 3 * mc["daily_se"] + 0.01
print(f"shifted barrier {shifted:.2f}: formula {bgk:.4f}; daily Monte Carlo {mc['daily']:.4f} ± {mc['daily_se']:.4f}")

**Exercise 5 (In code).** Check in-out parity with QuantLib: up-and-in plus up-and-out equals the vanilla call.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
ui = ql_barrier(ql.Barrier.UpIn, BARRIER)
assert abs(ui + barrier["ql_up_out"] - barrier["vanilla"]) < 1e-8
print(f"up-and-in {ui:.4f} + up-and-out {barrier['ql_up_out']:.4f} = {ui + barrier['ql_up_out']:.4f}; vanilla {barrier['vanilla']:.4f}")

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")